In [ ]:
import sys
import os

# Thêm project root vào sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import evaluate

# 1. Tải dữ liệu

In [ ]:
# Tải tập dữ liệu đã xử lý
train_df = pd.read_csv('../data/processed_3labels/train.csv', encoding='utf-8')
val_df = pd.read_csv('../data/processed_3labels/val.csv', encoding='utf-8')
test_df = pd.read_csv('../data/processed_3labels/test.csv', encoding='utf-8')

print(f"Train samples: {len(train_df)}")
print(f"Val samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")
print(f"\nLabel distribution (train):")
print(train_df['emotion'].value_counts())

Train samples: 3150
Val samples: 450
Test samples: 900

Label distribution (train):
emotion
NEGATIVE    1400
POSITIVE    1050
NEUTRAL      700
Name: count, dtype: int64


In [ ]:
# Chuyển đổi sang HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

ds = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

Ds

DatasetDict({
    train: Dataset({
        features: ['text', 'emotion'],
        num_rows: 3150
    })
    validation: Dataset({
        features: ['text', 'emotion'],
        num_rows: 450
    })
    test: Dataset({
        features: ['text', 'emotion'],
        num_rows: 900
    })
})

# 2. Khởi tạo Model và Tokenizer

In [ ]:
# Model configuration
model_name = "vinai/phobert-base-v2"
num_labels = 3

# Label mapping
label2id = {'POSITIVE': 0, 'NEUTRAL': 1, 'NEGATIVE': 2}
id2label = {0: 'POSITIVE', 1: 'NEUTRAL', 2: 'NEGATIVE'}

print(f"Model: {model_name}")
print(f"Number of labels: {num_labels}")
print(f"Label mapping: {label2id}")

Model: vinai/phobert-base-v2
Number of labels: 3
Label mapping: {'POSITIVE': 0, 'NEUTRAL': 1, 'NEGATIVE': 2}


In [ ]:
# Tải tokenizer và model
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
)

c:\Seminar\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\khanh\.cache\huggingface\hub\models--vinai--phobert-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better

## 3. Tokenize Dataset

In [8]:
def preprocess_function(examples):
    # Tokenize text
    tokenized = tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )
    
    # Convert emotion labels to ids
    tokenized['label'] = [label2id[e] for e in examples['emotion']]
    
    return tokenized

# Apply preprocessing
encoded_ds = ds.map(preprocess_function, batched=True)
encoded_ds

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Map: 100%|██████████| 900/900 [00:00<00:00, 2082.71 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'emotion', 'input_ids', 'token_type_ids', 'attention_mask', 'label'],
        num_rows: 3150
    })
    validation: Dataset({
        features: ['text', 'emotion', 'input_ids', 'token_type_ids', 'attention_mask', 'label'],
        num_rows: 450
    })
    test: Dataset({
        features: ['text', 'emotion', 'input_ids', 'token_type_ids', 'attention_mask', 'label'],
        num_rows: 900
    })
})

## 4. Setup Training

In [9]:
# Load metrics
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )['accuracy']
    
    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average='weighted'
    )['f1']
    
    return {'accuracy': accuracy, 'f1': f1}

In [10]:
# Training arguments
training_args = TrainingArguments(
    output_dir='_models/phobert-3-labels',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_dir='_models/logs',
    logging_steps=50,
    save_total_limit=2,
)

In [11]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_ds['train'],
    eval_dataset=encoded_ds['validation'],
    compute_metrics=compute_metrics,
)

# 5. Train Model

In [12]:
# Start training
print("🚀 Starting training...")
train_result = trainer.train()
print("\n✅ Training completed!")

🚀 Starting training...


c:\Seminar\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# 6. Đánh giá trên tập test

In [ ]:
print("📊 Evaluating on test set...")
test_results = trainer.evaluate(encoded_ds['test'])

print("\n📈 Test Results:")
print(f"  Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"  F1-score: {test_results['eval_f1']:.4f}")

# 7. Lưu model

In [ ]:
# Lưu model tốt nhất
final_model_path = '_models/phobert-3-labels-final'
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"✅ Model saved to: {final_model_path}")

# 8. Kiểm tra dự đoán

In [ ]:
# Kiểm tra với các câu test
test_sentences = [
    "Sản phẩm rất tốt, tôi rất hài lòng!",
    "Tệ quá, không đáng tiền",
    "Sản phẩm bình thường", 
    "Quá tuyệt vời! Mình rất thích",
    "Thất vọng lắm"
]

print("Kiểm tra dự đoán:\n")
for text in test_sentences:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=256)
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=-1)
    pred_idx = probs.argmax().item()
    confidence = probs[0][pred_idx].item()
    
    print(f"Text: {text}")
    print(f"  → {id2label[pred_idx]} ({confidence*100:.1f}%)\n")